In [1]:
import os
import sys
import pandas as pd
from pathlib import Path
import ast
import geopandas as gpd
import numpy as np
import matplotlib.pyplot as plt
import gc

# Change root directory to the repo root (Jupyter: __file__ is not defined)
def find_repo_root(start=Path.cwd()):
	for p in [start] + list(start.parents):
		if (p / 'Code').exists() or (p / '.git').exists() or (p / 'local_repo').exists():
			return p
	return start

repo_root = find_repo_root()
data_root = repo_root.parent.parent / 'Data' / 'CBOS ready'
geo_root = repo_root.parent.parent / 'Data' / 'Geospatial'
gus_root = Path(os.getcwd()).parent.parent.parent.parent / "Data" / "GUS"

os.chdir(repo_root)
sys.path.append(str(repo_root / 'Code' / 'tools'))

# import local toolkit (try normal import first, fall back to loading from file)
try:
	import inequality_analyzers as inqA
	import local_utility_functions as luf
except Exception:
	import importlib.util
	toolkit_path = repo_root / 'Code' / 'tools' / 'inequality_analyzers.py'
	if toolkit_path.exists():
		spec = importlib.util.spec_from_file_location("inequality_analyzers", str(toolkit_path))
		stk = importlib.util.module_from_spec(spec)
		spec.loader.exec_module(stk)
	else:
		raise


In [2]:
df_demographic = pd.read_csv(gus_root / "data" / 'bdl_demographic_data.csv', encoding='utf-8')
df_variables = pd.read_csv(gus_root / "metadata" / 'bdl_variables_level6.csv', encoding='utf-8')


In [3]:
def process_subject_data(subjectId):
    """
    Process demographic data for a given subject ID.
    Returns expanded dataframe with flattened structure.
    """
    # Filter by subjectId
    df_subject = df_demographic[df_demographic['subjectId'] == subjectId]
    
    # Get variable metadata for this subject
    variable_ids = df_subject['variableId'].unique()
    df_variables_subset = df_variables[df_variables['id'].isin(variable_ids)][['id', 'n1', 'n2', 'n3', 'n4', 'n5']]
    
    # Merge subject data with variables
    df_merged = pd.merge(df_subject, df_variables_subset, left_on='variableId', right_on='id', how='left')
    
    # Remove constant columns
    for col in ['n1', 'n2', 'n3', 'n4', 'n5']:
        if df_merged[col].nunique() <= 1:
            df_merged = df_merged.drop(columns=[col])
    
    # Parse values column
    def parse_values_column(value):
        try:
            value_list = ast.literal_eval(value)
            if isinstance(value_list, list) and len(value_list) == 1:
                return value_list[0]
            return value_list
        except (ValueError, SyntaxError):
            return None
    
    df_merged['values'] = df_merged['values'].apply(parse_values_column)
    
    # Expand and normalize
    df_expanded = df_merged.explode('values').reset_index(drop=True)
    values_normalized = pd.json_normalize(df_expanded['values'])
    df_expanded = pd.concat([df_expanded.drop(columns=['values']), values_normalized], axis=1)
    
    # Rename and format columns
    df_expanded = df_expanded.rename(columns={"id_x": "nuts_id", "id_y": "var_id"})
    df_expanded['nuts_id'] = df_expanded['nuts_id'].apply(lambda x: str(int(x)).zfill(12))
    df_expanded['teryt_id'] = df_expanded['nuts_id'].apply(luf.nuts_code_to_teryt)
    
    return df_expanded


In [4]:
df_var = {}
#subjects = list(set(df_demographic['subjectId']))
subjects = ['P2137']
for subject in subjects:
    df_var[subject] = process_subject_data(subject)
    df_var[subject]['teryt_id'] = df_var[subject]['nuts_id'].apply(luf.nuts_code_to_teryt)

gc.collect()

286

In [5]:
df_var['P2137']

,nuts_id,name,variableId,subjectId,var_id,n1,n2,year,val,attrId,teryt_id
0,000000000000,POLSKA,72305,P2137,72305,ogółem,ogółem,1995,38609399.0,1.0,0000000
1,000000000000,POLSKA,72305,P2137,72305,ogółem,ogółem,1996,38639341.0,1.0,0000000
2,000000000000,POLSKA,72305,P2137,72305,ogółem,ogółem,1997,38659979.0,1.0,0000000
3,000000000000,POLSKA,72305,P2137,72305,ogółem,ogółem,1998,38666983.0,1.0,0000000
4,000000000000,POLSKA,72305,P2137,72305,ogółem,ogółem,1999,38263303.0,1.0,0000000
...,...,...,...,...,...,...,...,...,...,...,...
7369075,071427338054,Wiskitki - miasto,454046,P2137,454046,0-14,kobiety,2024,123.0,1.0,1438054
7369076,071427338055,Wiskitki - obszar wiejski,454046,P2137,454046,0-14,kobiety,2021,701.0,1.0,1438055
7369077,071427338055,Wiskitki - obszar wiejski,454046,P2137,454046,0-14,kobiety,2022,704.0,1.0,1438055
7369078,071427338055,Wiskitki - obszar wiejski,454046,P2137,454046,0-14,kobiety,2023,686.0,1.0,1438055


In [6]:
df_census_1988 = pd.read_csv(gus_root / "data" / "census_data" / 'NSP1988_data.csv', encoding='utf-8')
df_census_2002 = pd.read_csv(gus_root / "data" / "census_data" / 'NSP2002_data.csv', encoding='utf-8')
df_census_2011 = pd.read_csv(gus_root / "data" / "census_data" / 'NSP2011_data.csv', encoding='utf-8')
df_census_2021 = pd.read_csv(gus_root / "data" / "census_data" / 'NSP2021_data.csv', encoding='utf-8')

df_census = {}
df_census[1988] = df_census_1988
df_census[2002] = df_census_2002
df_census[2011] = df_census_2011
df_census[2021] = df_census_2021

subjects = []
variables = []

for key in df_census.keys():
    subjects.append(list(df_census[key]['subjectId'].unique()))
    variables.append(list(df_census[key]['variableId'].unique()))
    
subjects = [v for var in subjects for v in var]
variables = [str(v) for var in variables for v in var]

In [7]:
print(subjects)
print(variables)

['P2884', 'P2885', 'P2883', 'P2887', 'P2114', 'P2403', 'P2402', 'P2871', 'P3304', 'P3311', 'P3309', 'P3310', 'P3420', 'P4253', 'P4320', 'P4345', 'P4287']
['196133', '196134', '196135', '196136', '196137', '196138', '196139', '196140', '196142', '196143', '196144', '196145', '196130', '196131', '196132', '196149', '196150', '196151', '196152', '9476', '9482', '9488', '9494', '9500', '9506', '9512', '9518', '9524', '9530', '9536', '9542', '9548', '9554', '9560', '9566', '9572', '9616', '9622', '9628', '9634', '9640', '9646', '9652', '9658', '9664', '9670', '9676', '9682', '9688', '9694', '9700', '9706', '9712', '9720', '9726', '9732', '9738', '9744', '9750', '9756', '9762', '9768', '9774', '9780', '9786', '9792', '9798', '9804', '9810', '9816', '46503', '46509', '46517', '60475', '60476', '60477', '60379', '60380', '60381', '60382', '60383', '60384', '60385', '60386', '60387', '60388', '60389', '60390', '60391', '60392', '60393', '60394', '60395', '60396', '60397', '60398', '60399', '604

In [8]:
# Set up API
# API key: c2ccd182-be71-41ee-6a99-08de61444905
# Another: 1de25faa-0796-4036-6a9c-08de61444905
# Another: 80b20037-8e82-42ec-6a9f-08de61444905
# Another: a2dbb3a9-026c-4aa5-6aa0-08de61444905
# Another: ed928d84-8050-4861-6aa1-08de61444905

import requests
from pandas import json_normalize
import time

#API_KEYS = ['1de25faa-0796-4036-6a9c-08de61444905', '80b20037-8e82-42ec-6a9f-08de61444905', 'a2dbb3a9-026c-4aa5-6aa0-08de61444905', 'ed928d84-8050-4861-6aa1-08de61444905', 'c2ccd182-be71-41ee-6a99-08de61444905']
API_KEYS = ["01fa8599-3e8c-48ef-6aa2-08de61444905"]

API_BASE = "https://bdl.stat.gov.pl/api/v1"
API_KEY = API_KEYS[0]
# os.getenv("BDL_API_KEY")  # optional: export BDL_API_KEY to raise rate limits

HEADERS = {"User-Agent": "LRDWI-Notebook/1.0"}
if API_KEY:
    HEADERS["X-ClientId"] = API_KEY

OUTPUT_DIR = gus_root / "metadata"

# Install nest_asyncio if not available
try:
    import nest_asyncio
except ImportError:
    import subprocess
    import sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "nest_asyncio", "aiohttp"])
    import nest_asyncio

In [11]:
# Jupyter-ready cell: download variable meta
# Produces: bdl_variables_level6.csv (full metadata returned by the API, flattened)
# Run this cell as-is. Optional: set API_KEY to your X-ClientId to raise rate limits.

# Install deps (uncomment if needed)
# %pip install requests pandas

ENDPOINT = f"{API_BASE}/variables"

session = requests.Session()
session.headers.update(HEADERS)

rows = []

print("Fetching variables (server-side filter: level=6)...")
i=1
for var in variables:
    params = {
        "format": "json",
        "lang": "pl"
    }
    endpoint = ENDPOINT + f"/{var}"
    print(endpoint)
    r = session.get(endpoint,params= params, timeout=30)
    r.raise_for_status()
    js = r.json()
    
    rows.append(js)
    print(f"Fetched variable {i}/{len(variables)}: {var}")
    time.sleep(0.25)
    i+=1


df = pd.DataFrame(rows)

# reorder some useful columns if present
cols_pref = [c for c in ["id","name","level","measureUnitId","measureUnitName","subjectId","subjectName"] if c in df.columns]
other_cols = [c for c in df.columns if c not in cols_pref]
df = df[cols_pref + other_cols]

# quick preview
df

# Save to CSV
output_path = OUTPUT_DIR / "census_meta.csv"
df.to_csv(output_path, index=False, encoding='utf-8')

Fetching variables (server-side filter: level=6)...
https://bdl.stat.gov.pl/api/v1/variables/196133
Fetched variable 1/755: 196133
https://bdl.stat.gov.pl/api/v1/variables/196134
Fetched variable 2/755: 196134
https://bdl.stat.gov.pl/api/v1/variables/196135
Fetched variable 3/755: 196135
https://bdl.stat.gov.pl/api/v1/variables/196136
Fetched variable 4/755: 196136
https://bdl.stat.gov.pl/api/v1/variables/196137
Fetched variable 5/755: 196137
https://bdl.stat.gov.pl/api/v1/variables/196138
Fetched variable 6/755: 196138
https://bdl.stat.gov.pl/api/v1/variables/196139
Fetched variable 7/755: 196139
https://bdl.stat.gov.pl/api/v1/variables/196140
Fetched variable 8/755: 196140
https://bdl.stat.gov.pl/api/v1/variables/196142
Fetched variable 9/755: 196142
https://bdl.stat.gov.pl/api/v1/variables/196143
Fetched variable 10/755: 196143
https://bdl.stat.gov.pl/api/v1/variables/196144
Fetched variable 11/755: 196144
https://bdl.stat.gov.pl/api/v1/variables/196145
Fetched variable 12/755: 1961

In [ ]:
df

,id,level,measureUnitId,measureUnitName,subjectId,n1,years,n2,n3
0,196133,7,26,osoba,P2884,ogółem,[1998],NaN,NaN
1,196134,7,26,osoba,P2884,0-9,[1998],NaN,NaN
2,196135,7,26,osoba,P2884,10-19,[1998],NaN,NaN
3,196136,7,26,osoba,P2884,20-29,[1998],NaN,NaN
4,196137,7,26,osoba,P2884,30-39,[1998],NaN,NaN
...,...,...,...,...,...,...,...,...,...
750,1652552,6,37,gosp.,P4287,gospodarstwa domowe 3-osobowe,[2021],NaN,NaN
751,1652553,6,37,gosp.,P4287,gospodarstwa domowe 4-osobowe,[2021],NaN,NaN
752,1652554,6,37,gosp.,P4287,gospodarstwa domowe 5-osobowe i większe,[2021],NaN,NaN
753,1652555,6,26,osoba,P4287,ludność w gospodarstwach domowych,[2021],NaN,NaN


In [ ]:
df_census_2002['subjectId'].unique()

In [ ]:
print(df_census_1988['subjectId'].unique())
print(df_census_1988['variableId'].unique())

In [ ]:
temp = pd.DataFrame({"id": [196133, 196134, 196135, 196136, 196137, 196138, 196139, 196140, ],
                     "level": [6,6,6,6,6,6,6,6, ],
                     "measureUnitId": [26,26,26,26,26,26,26,26,],
                     "measureUnitName": ["osoba","osoba","osoba","osoba","osoba","osoba","osoba","osoba",],
                     "subjectId": ["P2884","P2884","P2884","P2884","P2884","P2884","P2884","P2884",],
                     "n1": ["ogółem", "0-9", "10-19", "20-29", "30-39", "40-49", "50-59", "60 lat i więcej",],
                     "n2": [],
                     "n3": [],
                     "n4": [],
                     "n5": [],})

In [ ]:
df_variables[df_variables['subjectId'] == "P2114"]